In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import median_abs_deviation
import matplotlib.pyplot as plt
import seaborn as sns 
import geopandas as gpd
import contextily as ctx
from scipy.stats import chi2
from scipy.stats import zscore
from scipy.spatial import distance

df = pd.read_excel('base_usar.xlsx')
display(df.head(3))

In [ ]:
#IDs ENTEROS Y ORDENAMIENTO ESPACIAL
df['ID_Punto'] = df.groupby(['lon', 'lat'], sort=False).ngroup() + 1
df = df.sort_values(by=['ID_Punto', 'Lim_sup']).reset_index(drop=True)
df.to_excel('01_Base_IDs_Enteros.xlsx', index=False) #archivos id
print("-> Archivo generado: '01_Base_IDs_Enteros.xlsx'")

# Vemos cómo quedó el nuevo ID
display(df[['ID_Punto', 'lon', 'lat', 'Lim_sup', 'Lim_inf']].head(5))

In [ ]:

#comas a puntos
df['Lim_sup'] = df['Lim_sup'].astype(str).str.replace(',', '.').astype(float)
df['Lim_inf'] = df['Lim_inf'].astype(str).str.replace(',', '.').astype(float)
df['CO_g'] = df['CO_g'].astype(str).str.replace(',', '.').astype(float)

#identificar límites previos para comparación interna por ID_Punto
df['Lim_inf_anterior'] = df.groupby('ID_Punto')['Lim_inf'].shift(1)
df['Lim_sup_anterior'] = df.groupby('ID_Punto')['Lim_sup'].shift(1)

# detección de errores 
df['Es_Traslape'] = df['Lim_sup'] < df['Lim_inf_anterior']
df['Es_Repeticion'] = (df['Lim_sup'] == df['Lim_sup_anterior']) & (df['Lim_inf'] == df['Lim_inf_anterior'])
pozos_con_error = df[(df['Es_Traslape']) | (df['Es_Repeticion'])]['ID_Punto'].unique()
df_errores = df[df['ID_Punto'].isin(pozos_con_error)].copy()
df_errores.to_excel('02_Auditoria_Errores_Topologicos.xlsx', index=False)
print(f"generado: '02_Auditoria_Errores_Topologicos.xlsx'")
print(f"  Se detectaron {len(pozos_con_error)} pozos con problemas de traslape o repetición.")
df_limpio = df[~df['ID_Punto'].isin(pozos_con_error)].copy()
columnas_aux = ['Lim_inf_anterior', 'Lim_sup_anterior', 'Es_Traslape', 'Es_Repeticion']
df_limpio = df_limpio.drop(columns=columnas_aux)

df_limpio.to_excel('03_Base_Limpia_Fisica.xlsx', index=False)

print(f"generado: '03_Base_Limpia_Fisica.xlsx'")
print(f"   filas: {len(df_limpio)}")
display(df_limpio.head())

In [ ]:
df_final = pd.read_excel('03_Base_Limpia_Fisica.xlsx')

df_final['Intervalo'] = df_final['Lim_sup'].astype(int).astype(str) + ' - ' + df_final['Lim_inf'].astype(int).astype(str) + ' cm'
conteo_intervalos = df_final.groupby(['Lim_sup', 'Lim_inf', 'Intervalo']).size().reset_index(name='Frecuencia')
conteo_filtrado = conteo_intervalos[conteo_intervalos['Frecuencia'] > 10].copy()
conteo_filtrado = conteo_filtrado.sort_values(by=['Lim_sup', 'Lim_inf'], ascending=[True, True])
plt.figure(figsize=(10, 12)) 

# Usamos Seaborn para un renderizado limpio
ax = sns.barplot(
    data=conteo_filtrado,
    y='Intervalo',
    x='Frecuencia',
    color="#70A434" # El código hexadecimal de ese azul clásico
)

# 6. AÑADIR LAS ETIQUETAS DE TEXTO (Los numeritos al final de la barra)
for p in ax.patches:
    width = p.get_width()
    # Solo agregamos el texto si el ancho es mayor a 0
    if width > 0:
        plt.text(
            width + max(conteo_filtrado['Frecuencia']) * 0.01, # Un margen dinámico a la derecha
            p.get_y() + p.get_height() / 2, # Centrado verticalmente en la barra
            f'{int(width):,}', # Formato numérico con separador de miles
            va='center', 
            fontsize=9, 
            color='#333333'
        )


plt.title('Frecuencia de registros de Carbono Orgánico del Suelo (COS)\nordenados por profundidad (n > 10)', fontsize=14, weight='bold', pad=20)
plt.xlabel('Número de registros de mediciones de SOC (n)', fontsize=12, labelpad=10)
plt.ylabel('Intervalo de profundidad (cm)', fontsize=12, labelpad=10)

plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.4)
sns.despine(right=True, top=True)

plt.tight_layout()
plt.show()
df_final = df_final.drop(columns=['Intervalo'])

In [ ]:
#OUTLIERS UNIVARIADOS
df_limpio = pd.read_excel('03_Base_Limpia_Fisica.xlsx')
soc = df_limpio['CO_g']
#Q
Q1 = soc.quantile(0.25)
Q3 = soc.quantile(0.75)
IQR = Q3 - Q1
bajo_iqr = Q1 - 1.5 * IQR
alto_iqr = Q3 + 1.5 * IQR
df_limpio['Outlier_IQR'] = (soc < bajo_iqr) | (soc > alto_iqr)

# Z-Score
media = soc.mean()
desviacion = soc.std()
df_limpio['Z_Score'] = (soc - media) / desviacion
df_limpio['Outlier_ZScore'] = df_limpio['Z_Score'].abs() > 3

print(f"Atípicos por IQR: {df_limpio['Outlier_IQR'].sum()}")
print(f"Atípicos por Z-Score: {df_limpio['Outlier_ZScore'].sum()}")

In [ ]:
#MAHALANOBIS


#columna de profundidad media para el análisis
df_limpio['Prof_Media'] = (df_limpio['Lim_sup'] + df_limpio['Lim_inf']) / 2

#matriz de covarianza
data_mah = df_limpio[['lon', 'lat', 'Prof_Media', 'CO_g']].values

# Calculamos medias y matriz de covarianza inversa
centroide = np.mean(data_mah, axis=0)
matriz_cov = np.cov(data_mah.T)
inv_cov = np.linalg.inv(matriz_cov)

# Función para calcular la distancia de cada punto
def calcular_mahalanobis(x):
    dist = x - centroide
    return np.sqrt(np.dot(np.dot(dist, inv_cov), dist.T))

df_limpio['Dist_Mahalanobis'] = [calcular_mahalanobis(p) for p in data_mah]

# Umbral basado en distribución Chi-cuadrado (p < 0.001 para 4 variables)
umbral = chi2.ppf((1 - 0.001), df=4)
df_limpio['Outlier_Mahalanobis'] = df_limpio['Dist_Mahalanobis'] > umbral

# Exportamos la matriz final de diagnóstico
df_limpio.to_excel('04_Diagnostico_Outliers_Ecuador.xlsx', index=False)
print(f"Archivo '04_Diagnostico_Outliers_Ecuador.xlsx' generado.")
print(f"Atípicos detectados por Mahalanobis: {df_limpio['Outlier_Mahalanobis'].sum()}")

In [ ]:
print("Resumen estadístico de tus Distancias de Mahalanobis:")
print(df_limpio['Dist_Mahalanobis'].describe())

#umbrales menos estrictos
umbral_99 = chi2.ppf((1 - 0.01), df=4)  # 99% de confianza (~13.28)
umbral_95 = chi2.ppf((1 - 0.05), df=4)  # 95% de confianza (~9.49)

print(f"\nSi bajamos la severidad al 99% (Umbral > {umbral_99:.2f}):")
print(f"Detectaríamos {(df_limpio['Dist_Mahalanobis'] > umbral_99).sum()} atípicos.")
print(f"\nSi bajamos la severidad al 95% (Umbral > {umbral_95:.2f}):")
print(f"Detectaríamos {(df_limpio['Dist_Mahalanobis'] > umbral_95).sum()} atípicos.")

# puntos más "raros" del país
top_10_raros = df_limpio.nlargest(10, 'Dist_Mahalanobis')
display(top_10_raros[['ID_Punto', 'lon', 'lat', 'Lim_sup', 'Lim_inf', 'CO_g', 'Dist_Mahalanobis']])

In [ ]:
pip install geopandas contextily


In [ ]:
df_diag = pd.read_excel('04_Diagnostico_Outliers_Ecuador.xlsx')
gdf = gpd.GeoDataFrame(
    df_diag, 
    geometry=gpd.points_from_xy(df_diag.lon, df_diag.lat),
    crs="EPSG:4326"
) #conversion a datos geográficos


# pltmapa figura
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
metodos = ['Outlier_IQR', 'Outlier_ZScore', 'Outlier_Mahalanobis']
titulos = ['Mapa 1: Rango Intercuartílico (IQR)', 
           'Mapa 2: Z-Score', 
           'Mapa 3: Mahalanobis']

for i, metodo in enumerate(metodos):
    ax = axes[i]
    
    #puntos normales
    gdf[gdf[metodo] == False].plot(ax=ax, color='green', markersize=5, alpha=0.3, label='Normal')
    
    #atípicos
    atipicos = gdf[gdf[metodo] == True]
    
    if not atipicos.empty: #por si no hay atípicos
        atipicos.plot(ax=ax, color='red', markersize=40, alpha=0.8, edgecolor='black', label='Atípico')
    else:
        print(f"Nota: El {titulos[i]} no detectó ningún atípico para graficar.")
    
    # Fondo del mapa
    ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.5)

    ax.set_title(titulos[i], fontsize=14, fontweight='bold')
    ax.set_axis_off()
    ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

## correr el código de R


In [ ]:
#una vez con el archivo de r
df = pd.read_excel('05_Matriz_Ecuador_Armonizada_Final.xlsx')
gdf = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df.lon, df.lat),
    crs="EPSG:4326"
)

capas = ['000_005_cm', '005_015_cm', '015_030_cm', '030_060_cm', '060_100_cm']
titulos = ['0-5 cm', '5-15 cm', '15-30 cm', '30-60 cm', '60-100 cm']

#mapas
fig, axes = plt.subplots(1, 5, figsize=(25, 10), sharey=True)
fig.suptitle('Distribución Espacial y Estratificación de Carbono Orgánico (SOC) - Ecuador', 
             fontsize=22, fontweight='bold', y=0.98)


cmap = "YlOrBr"

for i, capa in enumerate(capas):
    ax = axes[i]
    
    # 3. Graficar los puntos con mayor intensidad
    # Subimos markersize a 25 y alpha a 1.0 para máxima visibilidad
    gdf.plot(column=capa, ax=ax, cmap=cmap, markersize=25, 
             legend=False, alpha=1.0, edgecolor='black', linewidth=0.5)
    
    # 4. Mapa de fondo Verde (OpenStreetMap)
    ctx.add_basemap(ax, crs=gdf.crs.to_string(), 
                    source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.7)
    
    ax.set_title(f'Estrato: {titulos[i]}', fontsize=15, fontweight='bold')
    ax.set_axis_off()

vmin = df[capas].min().min()
vmax = df[capas].max().max()

sm = plt.cm.ScalarMappable(cmap=cmap, 
                           norm=plt.Normalize(vmin=vmin, vmax=vmax))
cbar = fig.colorbar(sm, ax=axes, orientation='horizontal', fraction=0.02, pad=0.05)
cbar.set_label('Concentración de Carbono Orgánico (g/kg)', fontsize=13, fontweight='bold')

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

In [ ]:

df_antes = pd.read_excel('04_Diagnostico_Outliers_Ecuador.xlsx')
df_despues = pd.read_excel('05_Matriz_Ecuador_Armonizada_Final.xlsx')
n_antes = len(df_antes)
iqr_antes = df_antes['Outlier_IQR'].sum() if 'Outlier_IQR' in df_antes else 0
z_antes = df_antes['Outlier_ZScore'].sum() if 'Outlier_ZScore' in df_antes else 0
mah_antes = df_antes['Outlier_Mahalanobis'].sum() if 'Outlier_Mahalanobis' in df_antes else 0

stats_tabla = [{
    'Fase': 'ANTES (Datos Crudos)',
    'Capa / Estrato': 'Perfil Completo (Mixto)',
    'Puntos Válidos': n_antes,
    'Outliers IQR (%)': round((iqr_antes / n_antes) * 100, 2),
    'Outliers Z-Score (%)': round((z_antes / n_antes) * 100, 2),
    'Outliers Mahal. (%)': round((mah_antes / n_antes) * 100, 2)
}]

capas = ['000_005_cm', '005_015_cm', '015_030_cm', '030_060_cm', '060_100_cm']
nombres_capas = ['0-5 cm', '5-15 cm', '15-30 cm', '30-60 cm', '60-100 cm']

for idx, capa in enumerate(capas):
    # Filtramos nulos temporalmente
    df_valid = df_despues[['ID_Punto', 'lon', 'lat', capa]].dropna().copy()
    n_puntos = len(df_valid)
    
    if n_puntos == 0:
        continue

    # Cálculos estadísticos (IQR, Z-Score, Mahalanobis)
    Q1 = df_valid[capa].quantile(0.25)
    Q3 = df_valid[capa].quantile(0.75)
    IQR = Q3 - Q1
    df_valid['Outlier_IQR'] = (df_valid[capa] < (Q1 - 1.5 * IQR)) | (df_valid[capa] > (Q3 + 1.5 * IQR))
    pct_iqr = (df_valid['Outlier_IQR'].sum() / n_puntos) * 100

    df_valid['Z_Score'] = zscore(df_valid[capa])
    df_valid['Outlier_ZScore'] = abs(df_valid['Z_Score']) > 3
    pct_z = (df_valid['Outlier_ZScore'].sum() / n_puntos) * 100

    df_valid['Outlier_Mahalanobis'] = False
    cov_matrix = np.cov(df_valid[['lon', 'lat', capa]].values.T)
    inv_cov_matrix = np.linalg.inv(cov_matrix)
    mean_distr = df_valid[['lon', 'lat', capa]].mean(axis=0)
    df_valid['Mahalanobis'] = df_valid.apply(
        lambda row: distance.mahalanobis(row[['lon', 'lat', capa]], mean_distr, inv_cov_matrix), axis=1
    )
    df_valid['Outlier_Mahalanobis'] = df_valid['Mahalanobis'] > 3
    pct_mah = (df_valid['Outlier_Mahalanobis'].sum() / n_puntos) * 100

    stats_tabla.append({
        'Fase': 'DESPUÉS (Armonizado)',
        'Capa / Estrato': nombres_capas[idx],
        'Puntos Válidos': n_puntos,
        'Outliers IQR (%)': round(pct_iqr, 2),
        'Outliers Z-Score (%)': round(pct_z, 2),
        'Outliers Mahal. (%)': round(pct_mah, 2)
    })

    # Conversión Geográfica
    gdf = gpd.GeoDataFrame(
        df_valid, 
        geometry=gpd.points_from_xy(df_valid.lon, df_valid.lat),
        crs="EPSG:4326"
    )
    fig, axes = plt.subplots(1, 3, figsize=(20, 8))
    fig.suptitle(f'Diagnóstico Atípicos | Profundidad: {nombres_capas[idx]} (n={n_puntos})', 
                 fontsize=18, fontweight='bold', y=1.02)

    metodos = ['Outlier_IQR', 'Outlier_ZScore', 'Outlier_Mahalanobis']
    titulos = [f'IQR ({round(pct_iqr, 1)}%)', 
               f'Z-Score ({round(pct_z, 1)}%)', 
               f'Mahalanobis ({round(pct_mah, 1)}%)']

    for i, metodo in enumerate(metodos):
        ax = axes[i]
        gdf[gdf[metodo] == False].plot(ax=ax, color='#006400', markersize=12, alpha=0.5, label='Normal')
        atipicos = gdf[gdf[metodo] == True]
        if not atipicos.empty:
            atipicos.plot(ax=ax, color='#FF0000', markersize=50, alpha=0.9, edgecolor='black', linewidth=1, label='Atípico')
        
        ctx.add_basemap(ax, crs=gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.5)

        ax.set_title(titulos[i], fontsize=14, fontweight='bold')
        ax.set_axis_off()
        ax.legend(loc='lower right', frameon=True, fontsize='small')

    plt.tight_layout()
    plt.show()

print("\n   TABLA COMPARATIVA: IMPACTO DE LA ARMONIZACIÓN")
df_resumen = pd.DataFrame(stats_tabla)
print(df_resumen.to_string(index=False))
df_resumen.to_excel('06_Tabla_Comparativa_Outliers.xlsx', index=False)

In [ ]:
df = pd.read_excel('05_Matriz_Ecuador_Armonizada_Final.xlsx')
capas = ['000_005_cm', '005_015_cm', '015_030_cm', '030_060_cm', '060_100_cm']
gran_total = df[capas].count().sum()

print(f"Datos finales: {gran_total}")

In [ ]:


df_antes = pd.read_excel('04_Diagnostico_Outliers_Ecuador.xlsx')
df_despues = pd.read_excel('05_Matriz_Ecuador_Armonizada_Final.xlsx')
df_antes['Atipico_Global'] = (df_antes['Outlier_IQR'] | 
                              df_antes['Outlier_ZScore'] | 
                              df_antes['Outlier_Mahalanobis'])

gdf_antes = gpd.GeoDataFrame(
    df_antes, geometry=gpd.points_from_xy(df_antes.lon, df_antes.lat), crs="EPSG:4326"
)

capas = ['000_005_cm', '005_015_cm', '015_030_cm', '030_060_cm', '060_100_cm']
nombres_capas = ['0-5 cm', '5-15 cm', '15-30 cm', '30-60 cm', '60-100 cm']

colores_atipicos = ['#FF0000', '#FF4500', '#FF8C00', '#B22222', '#8B0000']

gdf_despues = gpd.GeoDataFrame(
    df_despues, geometry=gpd.points_from_xy(df_despues.lon, df_despues.lat), crs="EPSG:4326"
)

#VISUALIZACION
fig, axes = plt.subplots(1, 2, figsize=(22, 10))
fig.suptitle('Antes vs. Después de Splines', 
             fontsize=20, fontweight='bold', y=0.98)

#mapa 1
ax1 = axes[0]
# Normales en verde más intenso y grande
gdf_antes[gdf_antes['Atipico_Global'] == False].plot(ax=ax1, color='#228B22', markersize=15, alpha=0.5, label='Datos Normales')
# Atípicos en rojo fuerte
gdf_antes[gdf_antes['Atipico_Global'] == True].plot(ax=ax1, color='red', markersize=45, alpha=0.8, edgecolor='black', label='Atípicos Crudos')

ctx.add_basemap(ax1, crs=gdf_antes.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.5)
ax1.set_title("Base Cruda", fontsize=15, fontweight='bold')
ax1.legend(loc='lower right')
ax1.set_axis_off()

#mapa2
ax2 = axes[1]
# Perfil armonizado en verde más intenso y grande
gdf_despues.plot(ax=ax2, color='#006400', markersize=18, alpha=0.5, label='Perfil Armonizado')

# Graficar outliers por cada profundidad (mantenemos el bucle)
for i, capa in enumerate(capas):
    temp_df = df_despues[['ID_Punto', 'lon', 'lat', capa]].dropna().copy()
    if not temp_df.empty:
        # Z-Score robusto para detección
        z = np.abs(zscore(temp_df[capa]))
        out_idx = temp_df[z > 3].index
        
        if not out_idx.empty:
            gdf_despues.loc[out_idx].plot(ax=ax2, color=colores_atipicos[i], markersize=60, 
                                          edgecolor='black', alpha=1.0, label=f'Atípico {nombres_capas[i]}')

ctx.add_basemap(ax2, crs=gdf_despues.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.5)
ax2.set_title("Base Armonizada", fontsize=15, fontweight='bold')
ax2.legend(loc='lower right', title="Profundidad", fontsize=9)
ax2.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:

df = pd.read_excel('05_Matriz_Ecuador_Armonizada_Final.xlsx')
capas = ['000_005_cm', '005_015_cm', '015_030_cm', '030_060_cm', '060_100_cm']

df_clean = df.copy()

print(f"Registros iniciales: {len(df_clean)}")
capas_criticas = ['000_005_cm', '005_015_cm', '015_030_cm']

indices_a_eliminar = set()

for capa in capas_criticas:
    temp_df = df_clean[['ID_Punto', capa, 'lon', 'lat']].dropna()

    Q1 = temp_df[capa].quantile(0.25)
    Q3 = temp_df[capa].quantile(0.75)
    IQR = Q3 - Q1
    out_iqr = temp_df[(temp_df[capa] < (Q1 - 1.5 * IQR)) | (temp_df[capa] > (Q3 + 1.5 * IQR))].index

    z = np.abs(zscore(temp_df[capa]))
    out_z = temp_df[z > 3].index
    cov_matrix = np.cov(temp_df[['lon', 'lat', capa]].values.T)
    inv_cov_matrix = np.linalg.inv(cov_matrix)
    mean_distr = temp_df[['lon', 'lat', capa]].mean(axis=0)
    mah_dist = temp_df.apply(lambda row: distance.mahalanobis(row[['lon', 'lat', capa]], mean_distr, inv_cov_matrix), axis=1)
    out_mah = temp_df[mah_dist > 3].index
    
    indices_a_eliminar.update(out_iqr)
    indices_a_eliminar.update(out_z)
    indices_a_eliminar.update(out_mah)

df_final = df_clean.drop(list(indices_a_eliminar))

print(f"Atípicos detectados y removidos: {len(indices_a_eliminar)}")
print(f"Registros finales limpios: {len(df_final)}")
df_final.to_excel('07_Matriz_Ecuador_SIN_OUTLIERS_Final.xlsx', index=False)
